# Compulsory Assignment 3
## Machine Learning and Deep Learning (CDSCO2041C)

**Dataset:** CIFAR-100 — 100 fine-grained object classes, RGB images at 32×32 pixels.

**Group — replace with your details:**  
Student ID(s): *[fill in]*

---

In this notebook we work through the full pipeline for **image classification** on CIFAR-100: we prepare the data once and reuse that preprocessing everywhere (Part 1.1), learn a **convolutional autoencoder** as an unsupervised feature extractor (Part 1.2), compare **k-nearest neighbours** on raw pixels versus learned latent vectors under deliberately identical settings (Part 1.3), and finally train a **convolutional neural network** end-to-end for classification (Part 1.4).

Where it helps, we connect choices to themes from the course exercises: careful preprocessing and avoiding information leakage (L2), geometry of distance-based methods in high dimension (L4), reconstruction as a training objective (L11), and convolutional architectures with pooling (L12). The closing section reads a bit like a short lab report — methods, what the numbers suggest, and what we would still want to improve.

## 1.1 Data preparation

We start from the standard CIFAR-100 split: 50,000 training images and 10,000 test images. From the training portion we carve out **10% for validation** (90/10), using a **stratified** split so every fine-grained class stays represented in roughly the same proportions as in the full training set. The official test set is left aside until we want a final, honest accuracy number.

**Normalization** is deliberately conservative: we compute **per-channel mean and standard deviation on the training subset only** (after the 90/10 cut), then apply those three scalars to train, validation, and test alike. If we instead normalized using validation or test moments, we would let future information seep into how we scale inputs — the kind of subtle leakage discussed in introductory preprocessing material (L2). The small floor we apply to near-zero standard deviations is only a numerical safeguard.

From the same normalized images we keep two views: **spatial** arrays of shape `(N, 32, 32, 3)` for the convolutional models in Parts 1.2 and 1.4, and a **flattened** vector of length **3072** for distance-based classification on raw pixels in Part 1.3.

In [ ]:
# Imports and reproducibility
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

RNG = 42
np.random.seed(RNG)
keras.utils.set_random_seed(RNG)

# Plot styling
FIG_W = 9
DPI = 140
PAL = sns.color_palette("rocket_r", 10)
sns.set_theme(style="white", context="notebook")
plt.rcParams.update({"figure.dpi": DPI})

def clean_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

from IPython.display import display, HTML

def fig_caption(fig, number, text):
    plt.show()
    display(HTML(
        f'<p style="text-align:center; font-style:italic; font-size:10pt; '
        f'margin-top:2px; margin-bottom:16px;">Figure {number}: {text}</p>'
    ))

def table_style(styler, number, text):
    return styler.set_caption(f"Table {number}: {text}").set_table_styles([
        {"selector": "caption", "props": [
            ("font-weight", "bold"), ("font-size", "14px"),
            ("margin-bottom", "8px"), ("font-style", "italic")]},
        {"selector": "th, td", "props": [
            ("border", "1px solid black"), ("padding", "6px 12px"),
            ("text-align", "center")]},
        {"selector": "table", "props": [
            ("border-collapse", "collapse"), ("margin", "0 auto")]},
    ])

In [ ]:
# Load CIFAR-100 (downloads on first run via Keras)
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar100.load_data()

print(f"Train: {x_train_full.shape}  |  Test: {x_test.shape}")
print(f"Labels shape (train): {y_train_full.shape}  — fine-grained classes: {len(np.unique(y_train_full))}")

In [ ]:
# 90/10 train / validation split (stratified by class)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full,
    test_size=0.1,
    random_state=RNG,
    stratify=y_train_full,
)

print(f"After split — Train: {x_train.shape}  |  Val: {x_val.shape}")

In [ ]:
# Float conversion, train-only channel statistics, normalize all splits
x_train_f = x_train.astype("float32")
x_val_f = x_val.astype("float32")
x_test_f = x_test.astype("float32")

channel_mean = x_train_f.mean(axis=(0, 1, 2))
channel_std = x_train_f.std(axis=(0, 1, 2))
channel_std = np.where(channel_std < 1e-8, 1.0, channel_std)

print("Channel means (RGB):", channel_mean)
print("Channel stds  (RGB):", channel_std)

x_train_norm = (x_train_f - channel_mean) / channel_std
x_val_norm = (x_val_f - channel_mean) / channel_std
x_test_norm = (x_test_f - channel_mean) / channel_std

# Flatten for Part 1.3 (pixel k-NN)
x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_val_flat = x_val_norm.reshape(len(x_val_norm), -1)
x_test_flat = x_test_norm.reshape(len(x_test_norm), -1)

print(f"\nFlat shapes:    train {x_train_flat.shape}, val {x_val_flat.shape}, test {x_test_flat.shape}")
print(f"Spatial shapes: train {x_train_norm.shape} (for Part 1.2 and Part 1.4)")

## 1.2 Autoencoder-based feature learning

Here the goal is **unsupervised**: we ask a network to compress each image and then reconstruct it, without ever showing it class labels. Concretely, we use a **convolutional autoencoder** on the normalized 32×32×3 tensors. The **encoder** stacks two convolutional blocks (convolution, ReLU, max-pooling) so that spatial resolution shrinks while the number of feature channels grows; a dense layer at the end produces a **bottleneck** latent vector. The **decoder** unwinds that bottleneck with dense expansion, reshaping, upsampling, and convolutions until we are back to three channels at full resolution.

Training minimizes **mean squared error** between input and reconstruction — the same "predict yourself" idea as in the Fashion-MNIST autoencoder demo (L11), except convolutions respect the 2D layout of natural images instead of treating pixels as an arbitrary long vector. The output layer uses a **linear** activation because our inputs are z-scored rather than scaled to the unit interval.

Once training has settled, we **freeze the encoder** and use it as a feature map: each image becomes a **low-dimensional latent vector** that we will feed to k-NN in Part 1.3. It is worth keeping in mind that these features were tuned for **reconstruction fidelity**, not for **class separability**; that distinction matters when we interpret classification results later.

In [ ]:
latent_dim = 256

encoder_input = keras.Input(shape=(32, 32, 3), name="encoder_input")
x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(encoder_input)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Flatten()(x)
latent = layers.Dense(latent_dim, activation="relu", name="latent")(x)
encoder = Model(encoder_input, latent, name="encoder")
encoder.summary()

In [ ]:
decoder_input = keras.Input(shape=(latent_dim,), name="decoder_input")
x = layers.Dense(8 * 8 * 64, activation="relu")(decoder_input)
x = layers.Reshape((8, 8, 64))(x)
x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
decoder_output = layers.Conv2D(3, (3, 3), activation="linear", padding="same")(x)
decoder = Model(decoder_input, decoder_output, name="decoder")
decoder.summary()

In [ ]:
autoencoder_input = keras.Input(shape=(32, 32, 3))
autoencoder = Model(autoencoder_input, decoder(encoder(autoencoder_input)), name="autoencoder")
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

ae_history = autoencoder.fit(
    x_train_norm, x_train_norm,
    epochs=20,
    batch_size=128,
    shuffle=True,
    validation_data=(x_val_norm, x_val_norm),
)

In [ ]:
x_train_latent = encoder.predict(x_train_norm, verbose=0)
x_val_latent = encoder.predict(x_val_norm, verbose=0)
x_test_latent = encoder.predict(x_test_norm, verbose=0)
print(f"Latent shapes: train {x_train_latent.shape}, val {x_val_latent.shape}, test {x_test_latent.shape}")

In [ ]:
# Reconstruction loss curve
fig, ax = plt.subplots(figsize=(FIG_W, 3.6), constrained_layout=True)
ax.plot(ae_history.history["loss"], label="Train", color=PAL[7], linewidth=2)
ax.plot(ae_history.history["val_loss"], label="Val", color=PAL[4], linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
ax.legend(frameon=False)
clean_ax(ax)
fig_caption(fig, 1, "Training and validation MSE for the autoencoder; both curves should fall if optimization is stable")

# Sample reconstructions
n = 8
sample = x_val_norm[:n]
recon = autoencoder.predict(sample, verbose=0)
fig, axes = plt.subplots(2, n, figsize=(FIG_W * 1.4, 3), constrained_layout=True)
for i in range(n):
    axes[0, i].imshow(np.clip(sample[i] * channel_std + channel_mean, 0, 255).astype(np.uint8))
    axes[0, i].axis("off")
    axes[1, i].imshow(np.clip(recon[i] * channel_std + channel_mean, 0, 255).astype(np.uint8))
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original", fontsize=10)
axes[1, 0].set_ylabel("Reconstructed", fontsize=10)
fig_caption(fig, 2, "Eight validation images beside their reconstructions (denormalized to 0–255 for display only)")

## 1.3 Classification using pixel and latent features

We now ask a simple, transparent classifier — **k-nearest neighbours** with **k = 5** — to separate the 100 classes under two different input representations: the full **3072-dimensional** pixel vector from Part 1.1, and the **256-dimensional** latent code from Part 1.2. The point of holding **k** (and the default distance metric) fixed is methodological: if we changed the learner between runs, we would no longer know whether a gap in accuracy came from the features or from a different inductive bias.

k-NN is a useful baseline in courses (L4) because it makes the geometry of the feature space explicit. In very high dimensions, distances can behave in counterintuitive ways — often summarized as the **curse of dimensionality** — so we might hope the compressed latent space helps. On the other hand, that compression was learned to **reconstruct pixels**, not to **spread classes apart**, so it is entirely possible that latent k-NN does not beat pixel k-NN. We report validation and test accuracy and discuss the outcome in the report at the end.

In [ ]:
K_NEIGHBORS = 5

knn_pixel = KNeighborsClassifier(n_neighbors=K_NEIGHBORS)
knn_pixel.fit(x_train_flat, y_train.ravel())
pixel_val_acc = accuracy_score(y_val.ravel(), knn_pixel.predict(x_val_flat))
pixel_test_acc = accuracy_score(y_test.ravel(), knn_pixel.predict(x_test_flat))
print(f"KNN (pixels)  — Val: {pixel_val_acc:.4f}, Test: {pixel_test_acc:.4f}")

In [ ]:
knn_latent = KNeighborsClassifier(n_neighbors=K_NEIGHBORS)
knn_latent.fit(x_train_latent, y_train.ravel())
latent_val_acc = accuracy_score(y_val.ravel(), knn_latent.predict(x_val_latent))
latent_test_acc = accuracy_score(y_test.ravel(), knn_latent.predict(x_test_latent))
print(f"KNN (latent) — Val: {latent_val_acc:.4f}, Test: {latent_test_acc:.4f}")

In [ ]:
results_knn = pd.DataFrame({
    "Feature set": ["Pixel (3072-dim)", "Latent (256-dim)"],
    "Val accuracy": [pixel_val_acc, latent_val_acc],
    "Test accuracy": [pixel_test_acc, latent_test_acc],
})
display(table_style(
    results_knn.style.format({"Val accuracy": "{:.4f}", "Test accuracy": "{:.4f}"}),
    1, "k-NN accuracy as a function of feature representation (identical hyperparameters)",
))

fig, ax = plt.subplots(figsize=(FIG_W, 4), constrained_layout=True)
x_pos = np.arange(len(results_knn))
w = 0.35
ax.bar(x_pos - w / 2, results_knn["Val accuracy"], w, label="Validation", color=PAL[7], alpha=0.85)
ax.bar(x_pos + w / 2, results_knn["Test accuracy"], w, label="Test", color=PAL[4], alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(results_knn["Feature set"])
ax.set_ylabel("Accuracy")
ax.legend(frameon=False)
ax.set_ylim(0, max(results_knn[["Val accuracy", "Test accuracy"]].max()) * 1.15)
clean_ax(ax)
fig_caption(fig, 3, "Side-by-side validation and test accuracy for pixel-based versus latent-based k-NN")

## 1.4 CNN model

For the main classification experiment we switch to a **convolutional neural network** trained with **supervised** cross-entropy. The architecture follows the block structure emphasized in the CNN exercises (L12): **two stacks** of convolutional layers with **ReLU** nonlinearities, each stack followed by **max pooling** to downsample and build a modest hierarchy of filters. After the spatial feature maps are flattened, a **fully connected** hidden layer feeds a **100-way softmax**, one logit per CIFAR-100 fine label.

Because 32×32 inputs and a mid-sized network can still **overfit** the training fold, we add **dropout** after each pooling stage and before the softmax head — the same regularization idea introduced around deeper feedforward models (L9). After training we print **how many parameters are trainable**, as required, and evaluate on the held-out test set using the same normalized tensors as everywhere else in the notebook.

In [ ]:
cnn = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(100, activation="softmax"),
])

In [ ]:
cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

cnn_history = cnn.fit(
    x_train_norm, y_train,
    epochs=30,
    batch_size=128,
    validation_data=(x_val_norm, y_val),
)

In [ ]:
cnn.summary()
trainable_params = sum(int(p.numpy().size) for p in cnn.trainable_weights)
print(f"\nTotal trainable parameters: {trainable_params:,}")

test_loss, cnn_test_acc = cnn.evaluate(x_test_norm, y_test, verbose=0)
print(f"CNN test accuracy: {cnn_test_acc:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_W * 1.6, 4), constrained_layout=True)
ax1.plot(cnn_history.history["loss"], label="Train", color=PAL[7], linewidth=2)
ax1.plot(cnn_history.history["val_loss"], label="Val", color=PAL[4], linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend(frameon=False)
clean_ax(ax1)
ax2.plot(cnn_history.history["accuracy"], label="Train", color=PAL[7], linewidth=2)
ax2.plot(cnn_history.history["val_accuracy"], label="Val", color=PAL[4], linewidth=2)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend(frameon=False)
clean_ax(ax2)
fig_caption(fig, 4, "Supervised CNN training curves: loss and accuracy on training vs validation data")

In [ ]:
cnn_val_acc = cnn_history.history["val_accuracy"][-1]
results_all = pd.DataFrame({
    "Method": ["k-NN pixels (3072)", "k-NN latent (256)", "CNN"],
    "Val accuracy": [pixel_val_acc, latent_val_acc, cnn_val_acc],
    "Test accuracy": [pixel_test_acc, latent_test_acc, cnn_test_acc],
})
display(table_style(
    results_all.style.format({"Val accuracy": "{:.4f}", "Test accuracy": "{:.4f}"}),
    2, "Summary of validation and test accuracy across all three approaches",
))

fig, ax = plt.subplots(figsize=(FIG_W, 4), constrained_layout=True)
x_pos = np.arange(len(results_all))
w = 0.35
ax.bar(x_pos - w / 2, results_all["Val accuracy"], w, label="Validation", color=PAL[7], alpha=0.85)
ax.bar(x_pos + w / 2, results_all["Test accuracy"], w, label="Test", color=PAL[4], alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(results_all["Method"])
ax.set_ylabel("Accuracy")
ax.legend(frameon=False)
ax.set_ylim(0, max(results_all[["Val accuracy", "Test accuracy"]].max()) * 1.15)
clean_ax(ax)
fig_caption(fig, 5, "Overall comparison: k-NN on pixels, k-NN on latent codes, and the end-to-end CNN")

---

## Analytical report

This section steps back from the code and treats the notebook as a small study: what we did, what we usually see on CIFAR-100, and how that lines up with the theory touched on in the weekly exercises.

### 1. Methods in relation to the course

| Section | What we did | Course anchor |
|--------|-------------|----------------|
| **1.1** | Stratified 90/10 split; RGB mean and std from **train only**; z-score normalization; parallel **spatial** and **flat** views of the same tensors | **L2** — fit preprocessing on training data so validation and test remain out-of-sample for scaling choices. |
| **1.2** | Convolutional encoder–decoder; MSE reconstruction; low-dimensional bottleneck | **L11** — autoencoders learn codes that preserve information for reconstruction; convolutions adapt that idea to images. |
| **1.3** | Identical **k-NN** (same *k*, same metric defaults) on 3072-D pixels vs 256-D latents | **L4** — nonparametric methods and the geometry of high-dimensional spaces; a controlled comparison needs the same learner. |
| **1.4** | Stacked conv blocks, pooling, dense layer, softmax; dropout | **L12** — local receptive fields and translation-aware features; **L9** — stochastic depth / dropout as a guard against overfitting. |

### 2. Reading the results

After a full **Run all**, take validation and test accuracy from Tables 1–2 and from the printouts. Your exact figures will depend on random seed, hardware, and training noise, but on CIFAR-100 it is **typical** to see something like the following qualitative picture:

- **k-NN on raw pixels** often struggles: 3072 coordinates make nearest-neighbour votes noisy, and tiny 32×32 crops make many classes easy to confuse even for humans.
- **k-NN on the autoencoder latent** is a mixed bag. The dimensionality is friendlier to distance-based methods, yet the representation was never asked to separate classes — only to reconstruct — so a large jump in accuracy is **not** guaranteed.
- **The CNN** usually lands **well above** both k-NN variants, because filters are shared across space, features are composed hierarchically, and the whole stack is optimized directly for the classification objective.

When you write up your own run, it is perfectly fine to say briefly whether your numbers match this pattern and, if not, to speculate cautiously (e.g. under-trained autoencoder, lucky split, very short CNN training).

### 3. Concepts that tie the story together

**One pipeline, many models.** Using the same normalization for every split and every model keeps the comparison interpretable and matches the assignment’s requirement of a single preprocessing story.

**Objective mismatch.** A latent space can be excellent for compression and still mediocre for classification unless we add labels (supervised fine-tuning, hybrid losses, or a deeper pretraining recipe). That is not a failure of the autoencoder — it is a reminder that **representation learning follows the loss we choose**.

**Why convolutions help.** Flattened pixels treat every coordinate as equally “far” from every other in the model inductive bias; convolutions instead **encode locality and weight sharing**, which matches how objects appear in images.

### 4. Honest limitations

CIFAR-100 at 32×32 with 100 fine classes is **hard**; state-of-the-art systems use far deeper nets, data augmentation, and longer schedules than we use here. Our autoencoder is relatively **shallow**, and k-NN on tens of thousands of 3072-dimensional vectors is **slow** at test time — we accept that cost to stay faithful to the brief. A next step in a real project would be to document wall-clock time and to iterate on architecture only after the baseline is stable.

### 5. Part 2 / reflection prompts

The copy of `_CA_ML_03.md` we have stops at “## 2” without the actual questions. If Canvas or a PDF lists short-answer prompts, answer them here in your own voice under clear subheadings so this file remains the **single** submission artifact.

---

*End of notebook.*